# JEPA on recurrence representations: frozen-probe study

## 1. Setup, questions, and pre-registered comparisons

This study asks whether JEPA pretraining on recurrence representations improves frozen embeddings over raw-series JEPA, MAE, random initialization, and RQA features, and whether learned recurrence attention helps over external recurrence plots. All comparison rules are fixed before results are inspected. Metrics are accuracy for `system` and R² for the five regression tasks.

- **Q1:** Select the best of `rp_cnn`, `rp_s4nd`, and `rp_vit` by mean validation-probe metric across all six tasks and both seeds. Compare that selected RP-JEPA model with raw-series `raw_attn` JEPA on test, reporting the six task differences and their unweighted mean. Test labels are not used for selection.
- **Q2:** Compare exact `recattn` JEPA with external-RP `rp_cnn` JEPA and with `raw_attn` JEPA.
- **Q3:** For `raw_attn`, `rp_cnn`, and exact `recattn`, compare JEPA with MAE and random initialization using the same encoder specification.
- **Q4:** Compare every main-grid JEPA encoder with the Step 1 RQA-feature linear-probe baseline.

For every per-task comparison, the difference is method A minus method B. A paired bootstrap resamples the same probe-test windows for both methods (2,000 resamples); the table reports the bootstrap 95% percentile interval, the observed mean difference across seeds, and the sample SD of seed-level paired differences. The cross-task mean is the unweighted mean of the six task metrics.

In [ ]:
from pathlib import Path
import sys, os, json, time, hashlib, warnings
sys.path.insert(0, str(Path('../../src').resolve()))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, RidgeCV
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from rpjepa.data import TASKS
from rpjepa.representations import delay_embed, rp_input
from rpjepa.features import rqa_features
from rpjepa.encoders import make_encoder
from rpjepa.jepa import pretrain_jepa
from rpjepa.probe import embed, linear_probe
from rpjepa.train import random_init_encoder
from rpjepa.experiment import SMOKE, get_split

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N, M, TAU, D_MODEL, DEPTH = 64, 3, 1, 64, 2
LR = 1e-3
STEPS = 20 if SMOKE else 4000
BATCH = 32 if SMOKE else 128
N_PRE_TASK = 300 if SMOKE else 20000
PROBE_N = 300 if SMOKE else 5000
VAL_N = 300 if SMOKE else 2000
LOW_N = 100 if SMOKE else 500
SEEDS = (0,) if SMOKE else (0, 1)
BOOTSTRAPS = 2000
TABLE_ROOT = Path('../../results/tables/jepa/step3') / ('smoke' if SMOKE else '')
CACHE_ROOT = Path('../../data/processed/jepa/step3') / ('smoke' if SMOKE else '')
ENCODER_ROOT = CACHE_ROOT / 'encoders'
LOG_ROOT = CACHE_ROOT / 'logs'
for folder in (TABLE_ROOT, ENCODER_ROOT, LOG_ROOT): folder.mkdir(parents=True, exist_ok=True)
print({'device': DEVICE, 'smoke': SMOKE, 'N': N, 'pretrain_per_task': N_PRE_TASK, 'pool_total': 6*N_PRE_TASK, 'steps': STEPS, 'batch': BATCH, 'seeds': SEEDS})

## 2. Data and model grid

The unlabeled pretraining set is an equal mixture of 20,000 windows per task (120,000 total) from dedicated `pretrain` splits. Task labels and metadata are discarded; only each split's `x` array is concatenated. Recall pretraining uses four key–value pairs. Probe splits use the Step 1 split identities, including the default four-pair recall task.

In [ ]:
SPEC = {
    'raw_attn': ('raw_attn', {}), 'raw_conv': ('raw_conv', {}), 'raw_hyena': ('raw_hyena', {}),
    'rp_cnn': ('rp_cnn', {'rp_kind': 'D+series'}), 'rp_s4nd': ('rp_s4nd', {'rp_kind': 'D+series'}),
    'rp_vit': ('rp_vit', {'rp_kind': 'D+series', 'pos': 'grid'}),
    'recattn': ('recattn', {'exact': True}), 'recattn_taylor': ('recattn_taylor', {}),
}
MAIN_NAMES = ('raw_attn','raw_conv','raw_hyena','rp_cnn','rp_s4nd','rp_vit','recattn','recattn_taylor')
if SMOKE: MAIN_NAMES = ('raw_attn','rp_cnn','recattn')
def config(method, spec, objective='jepa', mask='future', enc_kw=None):
    enc_name, default_kw = SPEC[spec]
    return {'method': method, 'spec': spec, 'enc_name': enc_name, 'enc_kw': default_kw if enc_kw is None else enc_kw,
            'objective': objective, 'mask': mask}
main_configs = [config(f'jepa_{name}', name) for name in MAIN_NAMES]
mae_names = ('raw_attn','rp_cnn','recattn')
pretrain_configs = main_configs + [config(f'mae_{name}', name, objective='mae') for name in mae_names]
if SMOKE:
    ablation_configs = [config('jepa_rp_cnn_mask_cell_asym','rp_cnn',mask='cell_asym')]
else:
    ablation_configs = ([config(f'jepa_rp_cnn_mask_{mask}','rp_cnn',mask=mask) for mask in ('block','random','cell_sym','cell_asym')] +
        [config(f'jepa_rp_cnn_input_{tag}','rp_cnn',enc_kw={'rp_kind':kind}) for tag,kind in (('D','D'),('RPstack','RPstack+series'))] +
        [config('jepa_rp_vit_diag','rp_vit',enc_kw={'rp_kind':'D+series','pos':'diag'})])
all_configs = []
seen_config_keys = set()
for cfg in pretrain_configs + ablation_configs:
    identity = (cfg['method'], cfg['spec'], cfg['objective'], cfg['mask'], json.dumps(cfg['enc_kw'],sort_keys=True))
    if identity not in seen_config_keys: all_configs.append(cfg); seen_config_keys.add(identity)
print('Main grid:', len(main_configs), 'main JEPA configurations;', len(pretrain_configs)-len(main_configs), 'MAE configurations')
print('Ablation configurations:', len(ablation_configs), '| unique pretraining configurations:', len(all_configs))

pretrain_arrays = []
for task in TASKS:
    difficulty = {'n_pairs':4} if task == 'recall' else {}
    unlabeled = get_split(task, 'pretrain', N_PRE_TASK, N, M, TAU, base_seed=31415, **difficulty)
    pretrain_arrays.append(unlabeled.x)
x_pre = np.concatenate(pretrain_arrays, axis=0).astype(np.float32, copy=False)
del pretrain_arrays, unlabeled
print('Unlabeled x pool:', x_pre.shape, '| equal windows/task:', N_PRE_TASK)

## 3. Pretraining

Every configuration uses 4,000 updates, batch 128 and AdamW learning rate 1e-3: about 4 passes over the 120,000-window pool. This is a fixed working schedule, not a tuned optimum, and it is the same for every encoder, so comparisons are like for like. The collapse monitor flags runs whose final pooled-embedding spread is below 5 % of its initial value.

Checkpoints and stepwise loss/embedding-spread logs are stored under keys that include the full configuration; reruns load existing weights and logs.


In [ ]:
def cache_key(cfg, seed):
    return {'method':cfg['method'],'spec':cfg['spec'],'enc_name':cfg['enc_name'],'enc_kw':cfg['enc_kw'],
            'objective':cfg['objective'],'mask':cfg['mask'],'seed':int(seed),'N':N,'m':M,'tau':TAU,
            'd':D_MODEL,'depth':DEPTH,'steps':STEPS,'batch':BATCH,'lr':LR,
            'pretrain_n_per_task':N_PRE_TASK,'pretrain_task_order':list(TASKS)}
def key_digest(key): return hashlib.sha256(json.dumps(key,sort_keys=True,separators=(',',':')).encode()).hexdigest()[:20]
def pretrained_encoder(cfg, seed):
    key = cache_key(cfg,seed); stem = key_digest(key)
    state_path, log_path, meta_path = ENCODER_ROOT/f'{stem}.pt', LOG_ROOT/f'{stem}.csv', LOG_ROOT/f'{stem}.json'
    if state_path.exists() and log_path.exists() and meta_path.exists():
        meta = json.loads(meta_path.read_text(encoding='utf-8'))
        if meta.get('key') == key:
            model = make_encoder(cfg['enc_name'],D_MODEL,DEPTH,N,m=M,tau=TAU,**cfg['enc_kw'])
            model.load_state_dict(torch.load(state_path,map_location='cpu',weights_only=True)); model.eval()
            return model, pd.read_csv(log_path), float(meta['train_s']), True
    t0 = time.perf_counter()
    model, log = pretrain_jepa(cfg['enc_name'],D_MODEL,DEPTH,x_pre,n_states=N,m=M,tau=TAU,
        mask=cfg['mask'],steps=STEPS,batch=BATCH,lr=LR,seed=seed,device=DEVICE,enc_kw=cfg['enc_kw'],objective=cfg['objective'])
    train_s = time.perf_counter()-t0
    torch.save({name:value.detach().cpu() for name,value in model.state_dict().items()},state_path)
    log.to_csv(log_path,index=False)
    meta_path.write_text(json.dumps({'key':key,'train_s':train_s},sort_keys=True,indent=2),encoding='utf-8')
    return model, log, train_s, False

pretrain_rows, log_frames, trained_models = [], [], {}
for cfg in all_configs:
    for seed in SEEDS:
        model, log, train_s, loaded = pretrained_encoder(cfg,seed)
        model_key = (cfg['method'],seed)
        trained_models[model_key] = model
        initial_std = float(log.emb_std.iloc[0]); final_std = float(log.emb_std.iloc[-1])
        rate = STEPS/train_s if train_s > 0 else np.nan
        pretrain_rows.append({'method':cfg['method'],'spec':cfg['spec'],'encoder':cfg['enc_name'],
            'enc_kw':json.dumps(cfg['enc_kw'],sort_keys=True),'objective':cfg['objective'],'mask':cfg['mask'],
            'seed':seed,'steps':STEPS,'train_s':train_s,'steps_per_s':rate,'cache_loaded':loaded,
            'initial_emb_std':initial_std,'final_emb_std':final_std,
            'collapse_flag':bool(final_std < 0.05*initial_std)})
        curve = log.copy(); curve['method']=cfg['method']; curve['seed']=seed; log_frames.append(curve)
        print(f"{cfg['method']} seed={seed}: {rate:.2f} steps/s ({'cached' if loaded else 'trained'})")
pretrain_df = pd.DataFrame(pretrain_rows)
log_df = pd.concat(log_frames,ignore_index=True)
pretrain_df.to_csv(TABLE_ROOT/'pretrain_log.csv',index=False)
fig, axes = plt.subplots(1,2,figsize=(12,4))
for (method,seed), group in log_df.groupby(['method','seed']):
    axes[0].plot(group.step,group.loss,label=f'{method} s{seed}')
    axes[1].plot(group.step,group.emb_std,label=f'{method} s{seed}')
axes[0].set(xlabel='Update',ylabel='Pretraining loss',title='Loss curves')
axes[1].set(xlabel='Update',ylabel='Pooled embedding std',title='Embedding spread (collapse monitor)')
for ax in axes: ax.grid(alpha=.25); ax.legend(fontsize=6,ncol=2)
plt.tight_layout(); plt.show()
display(pretrain_df)

## 4. Frozen probes and validation-only selection

Each checkpoint is evaluated on the six matched Step 1 `probe_train` / `probe_test` splits. Validation probes train on `probe_train` and score on the separate `probe_val` split; they are used for Q1 selection only. Low-label probes use a deterministic 500-example subset of probe training data (100 in smoke). RQA features use the Step 1 feature cache when available and otherwise are computed from the same training/test windows.

In [ ]:
def get_probe_data(task, split, seed, n):
    # Omitting recall difficulty preserves Step 1's split identity; its generator default is n_pairs=4.
    return get_split(task,split,n,N,M,TAU,base_seed=seed)
def metric_from_prediction(y,pred,kind):
    y=np.asarray(y); pred=np.asarray(pred)
    if kind=='cls': return float(np.mean(pred==y))
    denom=np.square(y.astype(float)-np.mean(y.astype(float))).sum()
    return float(1-np.square(y.astype(float)-pred.astype(float)).sum()/denom) if denom>0 else float('nan')
def fit_probe_predict(Ztr,ytr,Zte,kind):
    scaler=StandardScaler(); A=scaler.fit_transform(Ztr); B=scaler.transform(Zte)
    if kind=='cls':
        cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=0)
        search=GridSearchCV(LogisticRegression(max_iter=2000,solver='lbfgs',random_state=0),
            {'C':np.logspace(-3,2,6)},cv=cv,scoring='accuracy',refit=True,n_jobs=1)
        search.fit(A,ytr); return search.predict(B)
    model=RidgeCV(alphas=np.logspace(-3,3,13)); model.fit(A,np.asarray(ytr,dtype=float)); return model.predict(B)
def evaluate_probe(method, model, task, seed, n_train=PROBE_N, n_val=VAL_N, n_test=PROBE_N, low_label=True):
    tr=get_probe_data(task,'probe_train',seed,n_train); te=get_probe_data(task,'probe_test',seed,n_test)
    va=get_probe_data(task,'probe_val',seed,n_val); batch=64 if SMOKE else 512
    ztr=embed(model,tr.x,batch=batch,device=DEVICE); zte=embed(model,te.x,batch=batch,device=DEVICE); zva=embed(model,va.x,batch=batch,device=DEVICE)
    test_result=linear_probe(ztr,tr.y,zte,te.y,tr.kind); val_result=linear_probe(ztr,tr.y,zva,va.y,tr.kind)
    test_pred=fit_probe_predict(ztr,tr.y,zte,tr.kind); val_pred=fit_probe_predict(ztr,tr.y,zva,tr.kind)
    rows=[{'method':method,'task':task,'seed':seed,'split':'probe_test','n_train':len(tr.x),**test_result},
          {'method':method,'task':task,'seed':seed,'split':'probe_val','n_train':len(tr.x),**val_result}]
    prediction_store[(method,task,seed,'probe_test')]=(te.y.copy(),np.asarray(test_pred).copy(),tr.kind)
    prediction_store[(method,task,seed,'probe_val')]=(va.y.copy(),np.asarray(val_pred).copy(),tr.kind)
    if low_label:
        rng=np.random.default_rng(9000+seed); chosen=rng.choice(len(tr.x),size=min(LOW_N,len(tr.x)),replace=False)
        low_result=linear_probe(ztr[chosen],tr.y[chosen],zte,te.y,tr.kind)
        low_pred=fit_probe_predict(ztr[chosen],tr.y[chosen],zte,tr.kind)
        low_rows.append({'method':method,'task':task,'seed':seed,'n_train':len(chosen),**low_result})
        prediction_store[(method,task,seed,'lowlabel')]=(te.y.copy(),np.asarray(low_pred).copy(),tr.kind)
    return rows

prediction_store={}; probe_rows=[]; low_rows=[]; evaluated_configs=[]
for cfg in all_configs:
    for seed in SEEDS:
        model=trained_models[(cfg['method'],seed)]
        for task in TASKS: probe_rows.extend(evaluate_probe(cfg['method'],model,task,seed))
    evaluated_configs.append(cfg)

# Random-init references for every main-grid encoder specification.
random_names=MAIN_NAMES
random_specs={name:SPEC[name] for name in random_names}
for name,(enc_name,enc_kw) in random_specs.items():
    method=f'random_{name}'
    for seed in SEEDS:
        model=random_init_encoder(enc_name,D_MODEL,DEPTH,n_states=N,m=M,tau=TAU,seed=seed,enc_kw=enc_kw,device=DEVICE)
        for task in TASKS: probe_rows.extend(evaluate_probe(method,model,task,seed))
probe_df=pd.DataFrame(probe_rows); low_df=pd.DataFrame(low_rows)
probe_df.to_csv(TABLE_ROOT/'probes.csv',index=False); low_df.to_csv(TABLE_ROOT/'probes_lowlabel.csv',index=False)
display(probe_df.groupby(['method','split']).metric.agg(['mean','std','count']).round(4))

## 5. Ablations and learned recurrence kernel

The RP-CNN mask comparison tests future, block, random, symmetric-cell, and asymmetric-cell masking. `cell_asym` masks one cell block without its mirror, leaving the mirror cell visible; this should ease the pretext task without necessarily improving probes. Input variants compare D+series (main), D-only, and RPstack+series. RP-ViT compares grid and diagonal-offset positions. Exact and Taylor recurrence attention are compared from the main grid. For exact `recattn`, the first block's learned Gaussian kernel is tested against negative Euclidean delay-state distance on 256 probe windows per task/head, with ε reported per head.

In [ ]:
# Bootstrap evaluation will compare paired per-window predictions, not independent aggregate scores.
ablation_rows=[]
for cfg in ablation_configs:
    if cfg in main_configs: continue
    for seed in SEEDS:
        final_loss=float(log_df[(log_df.method==cfg['method'])&(log_df.seed==seed)].sort_values('step').loss.iloc[-1])
        for task in TASKS:
            row=[r for r in probe_rows if r['method']==cfg['method'] and r['task']==task and r['seed']==seed and r['split']=='probe_test'][0]
            ablation_rows.append({'ablation':cfg['method'],'task':task,'seed':seed,'metric':row['metric'],'final_loss':final_loss})
# Make the full mask/input/position grid explicit in the table; main configurations provide the reference rows.
for cfg in main_configs:
    if cfg['method'] in ('jepa_rp_cnn','jepa_rp_vit','jepa_recattn','jepa_recattn_taylor'):
        for seed in SEEDS:
            final_loss=float(log_df[(log_df.method==cfg['method'])&(log_df.seed==seed)].sort_values('step').loss.iloc[-1])
            for task in TASKS:
                row=[r for r in probe_rows if r['method']==cfg['method'] and r['task']==task and r['seed']==seed and r['split']=='probe_test'][0]
                ablation_rows.append({'ablation':cfg['method']+'_reference','task':task,'seed':seed,'metric':row['metric'],'final_loss':final_loss})
ablations_df=pd.DataFrame(ablation_rows)

recattn_cfg=next(c for c in main_configs if c['spec']=='recattn')
kernel_model=trained_models[(recattn_cfg['method'],SEEDS[0])].to(DEVICE).eval()
kernel_rows=[]
with torch.no_grad():
    for task in TASKS:
        td=get_probe_data(task,'probe_test',SEEDS[0],max(256, min(PROBE_N,256)))
        correlations=[]
        for start in range(0,256,16):
            xb=torch.as_tensor(td.x[start:start+16],dtype=torch.float32,device=DEVICE)
            kernels=kernel_model.kernel_matrix(xb,exact=True).float().cpu().numpy()
            states=delay_embed(xb,M,TAU).float()
            distances=torch.cdist(states,states).cpu().numpy()
            upper=np.triu_indices(N,1)
            for b in range(len(kernels)):
                for head in range(kernels.shape[1]):
                    correlations.append((head,float(spearmanr(kernels[b,head][upper],-distances[b][upper]).statistic)))
        eps=kernel_model.blocks[0].eps_log.detach().exp().cpu().numpy()
        for head in range(len(eps)):
            vals=np.asarray([v for h,v in correlations if h==head],dtype=float)
            kernel_rows.append({'task':task,'head':head,'n_windows':256,'spearman_mean':float(np.nanmean(vals)),
                'spearman_sd':float(np.nanstd(vals,ddof=1)),'epsilon':float(eps[head])})
recattn_kernel_df=pd.DataFrame(kernel_rows)

# Two qualitative examples: a chaotic Lyapunov window and an associative-recall window.
chaotic=get_probe_data('lyapunov','probe_test',SEEDS[0],PROBE_N).x[:1]
recall=get_probe_data('recall','probe_test',SEEDS[0],PROBE_N).x[:1]
fig,axes=plt.subplots(2,3,figsize=(10,6))
for row,(title,x) in enumerate((('Chaotic (Lyapunov)',chaotic),('Recall',recall))):
    xb=torch.as_tensor(x,dtype=torch.float32,device=DEVICE)
    with torch.no_grad(): kernel=kernel_model.kernel_matrix(xb,exact=True)[0].mean(0).cpu().numpy()
    euclidean=rp_input(xb,M,TAU,kind='D')[0,0].cpu().numpy()
    axes[row,0].plot(x[0],lw=.8); axes[row,0].set_title(title+' window')
    axes[row,1].imshow(euclidean,origin='lower',cmap='viridis'); axes[row,1].set_title('Euclidean D RP')
    axes[row,2].imshow(kernel,origin='lower',cmap='viridis'); axes[row,2].set_title('Learned Gaussian kernel')
plt.tight_layout(); plt.show()
display(ablations_df); display(recattn_kernel_df)

## 6. Paired comparisons, summaries, and run-time accounting

Q1 selects the RP-JEPA candidate using the mean validation score across tasks and both seeds, then compares selected RP-JEPA against raw attention on test. Q2–Q4 are test comparisons. The Step 1 supervised ceiling, random-init, and RQA rows are added to the summary if their Step 1 CSVs are present; absent inputs are reported explicitly.

In [ ]:
def bootstrap_difference(method_a,method_b,task,n_resamples=BOOTSTRAPS):
    seed_diffs=[]; boot_by_seed=[]
    for seed in SEEDS:
        a=prediction_store.get((method_a,task,seed,'probe_test')); b=prediction_store.get((method_b,task,seed,'probe_test'))
        if a is None or b is None: continue
        ya,pa,kind=a; yb,pb,_=b
        if not np.array_equal(ya,yb): raise ValueError(f'Unpaired test windows: {task}, seed {seed}')
        seed_diffs.append(metric_from_prediction(ya,pa,kind)-metric_from_prediction(yb,pb,kind))
        n=len(ya); rng=np.random.default_rng(12345+seed); reps=[]
        for offset in range(0,n_resamples,100):
            size=min(100,n_resamples-offset); ix=rng.integers(0,n,size=(size,n))
            if kind=='cls':
                da=(pa==ya).astype(float)-(pb==yb).astype(float); reps.extend(da[ix].mean(axis=1))
            else:
                yy=ya.astype(float); p1=np.asarray(pa,dtype=float); p2=np.asarray(pb,dtype=float)
                ys=yy[ix]; mean=ys.mean(axis=1,keepdims=True); denom=np.square(ys-mean).sum(axis=1)
                numerator=np.square(ys-p2[ix]).sum(axis=1)-np.square(ys-p1[ix]).sum(axis=1)
                reps.extend(np.divide(numerator,denom,out=np.full(size,np.nan),where=denom>0))
        boot_by_seed.append(np.asarray(reps,dtype=float))
    if not seed_diffs: return {'difference':np.nan,'seed_sd':np.nan,'ci_low':np.nan,'ci_high':np.nan,'seeds':0}
    boot=np.nanmean(np.stack(boot_by_seed),axis=0)
    return {'difference':float(np.mean(seed_diffs)),'seed_sd':float(np.std(seed_diffs,ddof=1)) if len(seed_diffs)>1 else np.nan,
            'ci_low':float(np.nanpercentile(boot,2.5)),'ci_high':float(np.nanpercentile(boot,97.5)),'seeds':len(seed_diffs)}

# Validation-only choice: mean validation probe metric across all tasks and seeds.
rp_candidates=[f'jepa_{name}' for name in ('rp_cnn','rp_s4nd','rp_vit') if f'jepa_{name}' in set(probe_df.method)]
val=probe_df[(probe_df.split=='probe_val')&probe_df.method.isin(rp_candidates)]
means=val.groupby('method').metric.mean()
chosen=means.idxmax()
selection_df=pd.DataFrame([{'selected_rp_jepa':chosen,'mean_validation_metric':float(means.loc[chosen])}])
display(selection_df)

# RQA features: reuse the Step 1 feature cache where available, otherwise compute/cache the same features here.
step1_root=Path('../../data/processed/jepa') / ('smoke' if SMOKE else 'step1')
def feature_frame(task,split,seed,td):
    ident={'task':task,'split':split,'seed':int(seed),'n':len(td.x),'n_states':N,'m':M,'tau':TAU,'difficulty':{}}
    digest=hashlib.sha256(json.dumps(ident,sort_keys=True).encode()).hexdigest()[:16]
    old_path=step1_root/'features'/f'{task}_{split}_{digest}.csv'
    if old_path.exists(): return pd.read_csv(old_path)
    new_path=CACHE_ROOT/'rqa_features'/f'{task}_{split}_{seed}_{len(td.x)}.csv'; new_path.parent.mkdir(parents=True,exist_ok=True)
    if new_path.exists(): return pd.read_csv(new_path)
    frame=rqa_features(td.x,m=M,tau=TAU); frame.to_csv(new_path,index=False); return frame
for seed in SEEDS:
    for task in TASKS:
        tr=get_probe_data(task,'probe_train',seed,PROBE_N); te=get_probe_data(task,'probe_test',seed,PROBE_N)
        ftr=feature_frame(task,'probe_train',seed,tr); fte=feature_frame(task,'probe_test',seed,te)
        imp=SimpleImputer(strategy='median'); ztr=imp.fit_transform(ftr); zte=imp.transform(fte)
        result=linear_probe(ztr,tr.y,zte,te.y,tr.kind); pred=fit_probe_predict(ztr,tr.y,zte,tr.kind)
        method='rqa+linear'; probe_rows.append({'method':method,'task':task,'seed':seed,'split':'probe_test','n_train':len(tr.x),**result})
        prediction_store[(method,task,seed,'probe_test')]=(te.y.copy(),np.asarray(pred).copy(),tr.kind)
probe_df=pd.DataFrame(probe_rows); probe_df.to_csv(TABLE_ROOT/'probes.csv',index=False)

comparison_specs=[]
comparison_specs.append(('Q1 RP-JEPA selected vs TS-JEPA',chosen,'jepa_raw_attn'))
comparison_specs.extend([('Q2 recattn vs external RP-JEPA','jepa_recattn','jepa_rp_cnn'),
    ('Q2 recattn vs TS-JEPA','jepa_recattn','jepa_raw_attn')])
for enc in ('raw_attn','rp_cnn','recattn'):
    comparison_specs.extend([(f'Q3 JEPA vs MAE ({enc})',f'jepa_{enc}',f'mae_{enc}'),
        (f'Q3 JEPA vs random-init ({enc})',f'jepa_{enc}',f'random_{enc}'),
        (f'Q3 MAE vs random-init ({enc})',f'mae_{enc}',f'random_{enc}')])
for name in MAIN_NAMES: comparison_specs.append(('Q4 JEPA vs RQA features',f'jepa_{name}','rqa+linear'))
# Deduplicate repeated comparison pairs.
comparison_specs=list(dict.fromkeys(comparison_specs))
comparison_rows=[]
for question,a,b in comparison_specs:
    tasks=list(TASKS)
    task_values=[]
    for task in tasks:
        stats=bootstrap_difference(a,b,task)
        comparison_rows.append({'question':question,'method_a':a,'method_b':b,'task':task,**stats})
        task_values.append(stats['difference'])
    comparison_rows.append({'question':question,'method_a':a,'method_b':b,'task':'mean',
        'difference':float(np.nanmean(task_values)),'seed_sd':np.nan,'ci_low':np.nan,'ci_high':np.nan,'seeds':int(np.sum(np.isfinite(task_values)))})
comparisons_df=pd.DataFrame(comparison_rows)
comparisons_df.to_csv(TABLE_ROOT/'comparisons.csv',index=False)
ablations_df.to_csv(TABLE_ROOT/'ablations.csv',index=False)
recattn_kernel_df.to_csv(TABLE_ROOT/'recattn_kernel.csv',index=False)
display(comparisons_df)

# Method x task + cross-task mean summary, with Step 1 references when present.
test_scores=probe_df[probe_df.split=='probe_test'].copy()
summary_rows=[]
for (method,task),g in test_scores.groupby(['method','task']): summary_rows.append({'method':method,'task':task,'metric':g.metric.mean(),'seed_sd':g.metric.std(ddof=1)})
step1_table_root=Path('../../results/tables/jepa/step1') / ('smoke' if SMOKE else '')
step1_files={name:step1_table_root/f'step1_{name}.csv' for name in ('supervised','random_init','baselines')}
loaded_step1={}
for label,path in step1_files.items():
    if Path(path).exists(): loaded_step1[label]=pd.read_csv(path)
    else: print(f'MISSING Step 1 input: {path}')
for label,frame in loaded_step1.items():
    if label=='supervised': frame=frame.rename(columns={'encoder':'encoder'})
    if label=='random_init': pass
    for (method,task),g in frame.groupby(['method','task']):
        if 'metric' in g: summary_rows.append({'method':str(method),'task':task,'metric':g.metric.mean(),'seed_sd':g.metric.std(ddof=1)})
summary_long=pd.DataFrame(summary_rows)
mean_rows=summary_long.groupby('method').metric.mean().rename('metric').reset_index(); mean_rows['task']='mean'; mean_rows['seed_sd']=np.nan
summary_long=pd.concat([summary_long,mean_rows],ignore_index=True)
summary_long['value']=summary_long.apply(lambda r:f"{r.metric:.3f} ({r.seed_sd:.3f})" if pd.notna(r.seed_sd) else f"{r.metric:.3f}",axis=1)
summary_table=summary_long.pivot_table(index='method',columns='task',values='value',aggfunc='first').reindex(columns=list(TASKS)+['mean'])
display(summary_table); summary_long.to_csv(TABLE_ROOT/'summary.csv',index=False)

print('Measured pretraining time per configuration (seconds; cached runs report their original time):')
display(pretrain_df.groupby(['method']).agg(seeds=('seed','count'), train_s=('train_s','mean'), steps_per_s=('steps_per_s','mean')).round(1))
print('Tables:',TABLE_ROOT.resolve())


## 7. Findings

Full run: 18 pretraining configurations × 2 seeds (4,000 updates, batch 128, 120,000 unlabeled windows), then frozen linear probes on 5,000/5,000 windows per task. No run collapsed by the pre-registered flag. Figures are in `results/tables/jepa/step3/figures/`. Differences are A − B on test with paired-bootstrap 95 % intervals.

**Probe metric per task (mean of 2 seeds), selected rows**

| method | system | lyapunov | noise | changepoint | recall | forecast | mean |
|---|---|---|---|---|---|---|---|
| JEPA rp_cnn | 0.842 | 0.845 | 0.792 | 0.011 | 0.047 | 0.608 | **0.524** |
| random-init rp_cnn | 0.842 | 0.845 | 0.751 | −0.000 | 0.058 | 0.583 | 0.513 |
| JEPA raw_attn (TS-JEPA) | 0.814 | 0.740 | 0.716 | −0.008 | 0.003 | 0.594 | 0.477 |
| random-init raw_attn | 0.829 | 0.768 | 0.734 | −0.005 | 0.011 | 0.642 | 0.496 |
| JEPA recattn | 0.823 | 0.751 | 0.706 | −0.006 | 0.003 | 0.575 | 0.475 |
| RQA features + linear | 0.712 | 0.796 | 0.924 | −0.004 | −0.003 | 0.208 | 0.439 |
| supervised rp_cnn (Step 1 ceiling, d = 64) | 0.877 | 0.925 | 0.984 | 0.112 | 0.999 | 0.983 | 0.813 |

1. **Q1: RP-JEPA beats TS-JEPA on every task.** rp_cnn was chosen on validation probes (0.524, ahead of rp_s4nd 0.415 and rp_vit 0.380). On test it beats raw-attention JEPA by +0.048 on average, with every per-task interval above zero (lyapunov +0.105, noise +0.076).
2. **But most of that gap is the architecture, not the pretraining.** Untrained, the same rp_cnn already scores 0.513. JEPA adds only +0.011 on average: noise +0.042, forecast +0.025, changepoint +0.011, recall −0.011, system and lyapunov ≈ 0. For raw attention, JEPA pretraining *lowers* the probes compared with random init (−0.020; all six intervals at or below zero). For recurrence attention it changes nothing (−0.001).
3. **Q2: building the RP inside the model did not help here.** Exact recurrence-attention JEPA equals TS-JEPA (−0.001) and is clearly below external-RP JEPA (−0.049; every interval below zero). After pretraining its kernel is still an RP: the Spearman correlation with −(Euclidean distance) is 0.87–0.97 across tasks, and ε stayed near its initial value (≈ 4).
4. **Q3: JEPA ≥ MAE, but neither beats random init by much.** JEPA beats MAE for all three encoders (+0.007 to +0.018 on average). MAE is below random init for raw_attn (−0.027) and recattn (−0.018). Latent prediction is the better objective, but on this benchmark neither objective extracts much beyond what a random projection already provides.
5. **Q4: frozen embeddings vs RQA features.** RP-JEPA beats RQA + linear on average (+0.085), mostly on system (+0.130) and forecast (+0.400). RQA features stay far better on noise (0.924 vs 0.792). The other JEPA encoders are within ±0.04 of RQA on average.
6. **Frozen probes are far below supervised training.** The best frozen embedding reaches a mean of 0.52 vs 0.81 supervised. The gaps are biggest on recall (0.05 vs 0.999) and forecast. The same holds with 500 labels (RP-JEPA 0.461 vs random-init 0.442). The pretext task does not teach recall, because masking future states does not require finding the matching key.
7. **Ablations (rp_cnn).**
   - **Future masking is best.** Mean probe 0.524, vs block 0.512, random 0.507, symmetric cell 0.476, asymmetric cell 0.479.
   - **Cell masking leaks as expected.** Asymmetric cell masks leave the mirror cell visible, which lowers the pretext loss (0.097 vs 0.150 for symmetric) with no probe gain.
   - **Distance-only input makes the pretext trivial.** Loss falls to 0.002 and the embedding spread shrinks below its initial value; probes drop to 0.393, with forecast at 0.05.
   - **Thresholded RP stacks** cost a little (0.496).
   - **rp_vit:** diagonal-offset positions did not help (0.359 vs 0.374).

**Verdict for step 3.** JEPA on recurrence plots gives the best frozen embeddings of all the methods tested, but almost all of the advantage comes from the RP + CNN architecture being a good random feature extractor, not from the self-supervised pretraining.

**Caveats.**
- One schedule (4,000 updates, learning rate 1e-3) and one width (d = 64), not tuned.
- Two seeds.
- Pretraining pool and probe tasks come from the same generators.
